# Universidad Federico Santa María - 2026
## FINANZAS
### Profesor Fernando Díaz H.

# WACC: Aplicación a una Empresa que No Transa en Bolsa

## El caso: Chick-fil-A

**Chick-fil-A** es una de las cadenas de comida rápida más grandes de Estados Unidos (ventas del sistema cercanas a **USD 24 mil millones** en 2025). Sin embargo, es una empresa **privada**, controlada por la familia Cathy: **sus acciones no se transan en bolsa**.

Supongamos que el directorio evalúa un plan de expansión y necesita una **tasa de descuento** para los flujos de los nuevos locales. El problema es que **no podemos estimar el beta de Chick-fil-A**: no hay serie de precios de su acción.

La solución estándar es el **método de empresas comparables** (*pure play*): usamos empresas que sí transan en bolsa y que tienen un **riesgo operacional (de los activos)** similar.

### ¿Qué haremos?
1. Elegiremos un grupo de **empresas comparables** que transan en bolsa.
2. Descargaremos sus precios y estimaremos su **beta de mercado apalancado** ($\beta^L$) con el modelo de mercado.
3. **Desapalancaremos** cada beta usando su estructura de capital ($B/E$) para obtener el **beta de los activos** ($\beta^U$).
4. Combinaremos los $\beta^U$ de las comparables en un **beta de activos del sector**.
5. **Reapalancaremos** ese beta con la estructura de capital **objetivo** de Chick-fil-A.
6. Calcularemos el **costo del patrimonio** ($r_E$) con el CAPM y el **costo de la deuda** ($r_B$) con un **rating sintético**.
7. Calcularemos el **WACC** y analizaremos su **sensibilidad** al nivel de endeudamiento.

## Recordatorio de fórmulas (ver presentación *Estructura de Capital*)

**Beta apalancado y desapalancado** (Hamada, deuda libre de riesgo, con impuestos):
$$
\beta^L = \beta^U\left[1 + (1-\tau_c)\frac{B}{E}\right]
\qquad\Longleftrightarrow\qquad
\beta^U = \frac{\beta^L}{1 + (1-\tau_c)\,B/E}
$$

**Costo del patrimonio** (CAPM):
$$
r_E = r_f + \beta^L\,\big(\mathbb{E}[r_M] - r_f\big)
$$

**Costo de la deuda**:
$$
r_B = r_f + \text{spread de crédito}
$$

**Costo de capital promedio ponderado**:
$$
WACC = \frac{E}{B+E}\,r_E + \frac{B}{B+E}\,r_B\,(1-\tau_c)
$$

## Cargando las librerías

In [ ]:
%pip install -q yfinance pandas_datareader statsmodels

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import pandas_datareader.data as web
import statsmodels.api as sm
from IPython.display import display, Markdown

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Parámetros del caso

Todos los **supuestos** del análisis están en esta celda. Modifíquelos para ver cómo cambia el resultado.

| Parámetro | Valor | Comentario |
|---|---|---|
| `TICKERS` | MCD, YUM, QSR, DPZ, WING, WEN | Comparables: cadenas de comida rápida con modelo de franquicias |
| `MERCADO` | ^GSPC | S\&P 500 como proxy del portafolio de mercado |
| `INICIO`, `FIN` | ago-2021 a ago-2026 | 60 retornos mensuales (5 años), práctica estándar |
| `TAU` | 21% | Tasa legal federal de impuesto corporativo en EE.UU. |
| `W_B_OBJ` | 20% | **Supuesto**: estructura de capital objetivo de Chick-fil-A, $B/(B+E)$ |
| `EBIT_V` | 5% | **Supuesto**: EBIT / Valor de la firma (equivale a un múltiplo $V/EBIT \approx 20\times$, similar al de las comparables) |

**¿Por qué estas comparables?** Chick-fil-A es una cadena de comida rápida (*quick service restaurant*) que opera con franquiciados. Las comparables comparten ese negocio:

- **MCD** (McDonald's), **YUM** (KFC, Taco Bell, Pizza Hut), **QSR** (Restaurant Brands: Burger King, Popeyes, Tim Hortons), **DPZ** (Domino's), **WING** (Wingstop, especializada en pollo) y **WEN** (Wendy's).
- Excluimos cadenas *fast casual* con locales propios (por ejemplo Chipotle), cuyo riesgo operacional es distinto.

In [ ]:
EMPRESA  = "Chick-fil-A"
TICKERS  = ["MCD", "YUM", "QSR", "DPZ", "WING", "WEN"]
NOMBRES  = {"MCD": "McDonald's", "YUM": "Yum! Brands", "QSR": "Restaurant Brands Intl.",
            "DPZ": "Domino's Pizza", "WING": "Wingstop", "WEN": "Wendy's"}
MERCADO  = "^GSPC"
INICIO   = "2021-08-01"
FIN      = "2026-09-01"

TAU      = 0.21     # tasa de impuesto corporativo
W_B_OBJ  = 0.20     # B/(B+E) objetivo de la empresa privada (supuesto)
EBIT_V   = 0.05     # EBIT / Valor de la firma (supuesto, para el rating sintético)

## Estructura de capital de las comparables

Para desapalancar necesitamos la razón **deuda / patrimonio** de cada comparable, **a valor de mercado**:

- $E$: **capitalización bursátil** (precio × número de acciones).
- $B$: **deuda total**. Usamos el valor libro como aproximación del valor de mercado de la deuda (práctica habitual cuando la deuda no transa).

Los datos corresponden a **Yahoo Finance, *Key Statistics***, al 24-sep-2026 (deuda del último trimestre reportado, junio 2026). Cifras en **miles de millones de USD**.

> **Nota sobre arriendos:** desde la norma ASC 842, la "Total Debt" de Yahoo incluye los **pasivos por arriendo**. En restaurantes estos montos son relevantes. Incluirlos es consistente, siempre que tratemos igual a la empresa objetivo.

Si desea **actualizar los datos a hoy**, cambie `ACTUALIZAR = True` en la celda siguiente.

In [ ]:
estructura = pd.DataFrame({
    "E": [214.00, 38.38, 25.08, 9.81, 2.67, 1.30],   # capitalización bursátil (USD miles de millones)
    "B": [ 54.60, 13.41, 15.65, 5.12, 1.27, 4.07],   # deuda total (USD miles de millones)
}, index=TICKERS)

ACTUALIZAR = False   # True: descarga E y B actuales desde Yahoo Finance
if ACTUALIZAR:
    for t in TICKERS:
        info = yf.Ticker(t).info
        estructura.loc[t, "E"] = info["marketCap"] / 1e9
        estructura.loc[t, "B"] = info["totalDebt"] / 1e9

estructura["B/E"] = estructura["B"] / estructura["E"]
estructura["B/(B+E)"] = estructura["B"] / (estructura["B"] + estructura["E"])
estructura.insert(0, "Empresa", [NOMBRES[t] for t in TICKERS])
estructura

Observe la enorme dispersión: McDonald's tiene $B/E \approx 0{,}26$, mientras que **Wendy's** tiene $B/E > 3$. Si comparáramos directamente los betas apalancados, estaríamos mezclando **riesgo del negocio** con **riesgo financiero**. Por eso debemos desapalancar.

## Descarga de precios y cálculo de retornos

Descargamos precios **mensuales** (ajustados por dividendos y *splits*) de las comparables y del S\&P 500, y calculamos **retornos logarítmicos mensuales**:
$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

In [ ]:
precios = yf.download(TICKERS + [MERCADO], start=INICIO, end=FIN,
                      interval="1mo", auto_adjust=True, progress=False)["Close"]
precios = precios.dropna(how="all")[TICKERS + [MERCADO]]
retornos = np.log(precios).diff().dropna()

print(f"Período: {retornos.index[0]:%Y-%m} a {retornos.index[-1]:%Y-%m}  ({len(retornos)} meses)")
retornos.tail()

In [ ]:
# Evolución de un dólar invertido en cada acción y en el S&P 500
acum = np.exp(retornos.cumsum())
ax = acum.drop(columns=MERCADO).plot(figsize=(10, 5), lw=1.2)
acum[MERCADO].plot(ax=ax, color="black", lw=2.5, label="S&P 500")
ax.set_title("Valor de USD 1 invertido")
ax.set_ylabel("USD")
ax.legend(ncol=4)
ax.grid(alpha=0.3)
plt.show()

## Paso 1: Beta apalancado de cada comparable

Estimamos el **modelo de mercado** por MCO para cada acción $i$:
$$
r_{i,t} = a_i + \beta^L_i\, r_{M,t} + \varepsilon_{i,t}
$$

El $\hat\beta^L_i$ estimado mide el riesgo sistemático del **patrimonio** de cada empresa, que incluye tanto el riesgo del negocio como el riesgo financiero de su deuda.

In [ ]:
filas = []
for t in TICKERS:
    X = sm.add_constant(retornos[MERCADO])
    modelo = sm.OLS(retornos[t], X).fit()
    filas.append({"Ticker": t,
                  "beta_L": modelo.params[MERCADO],
                  "e.e.": modelo.bse[MERCADO],
                  "R2": modelo.rsquared,
                  "N": int(modelo.nobs)})
betas = pd.DataFrame(filas).set_index("Ticker")
betas

**Interpretación.** Los restaurantes de comida rápida son en general **defensivos**: sus betas son menores a 1, porque la demanda por comida barata se mantiene relativamente estable en el ciclo económico. Wingstop es la excepción: su beta alto refleja una acción de **crecimiento** muy volátil. Los $R^2$ son bajos (15–20%): la mayor parte del riesgo de cada acción es **idiosincrático** (diversificable).

## Paso 2: Desapalancar

Con la estructura de capital de cada comparable, obtenemos su **beta de activos** (beta desapalancado):
$$
\beta^U_i = \frac{\beta^L_i}{1 + (1-\tau_c)\,(B/E)_i}
$$

El beta de activos mide solo el **riesgo del negocio**, que es lo que Chick-fil-A tiene en común con las comparables.

In [ ]:
comp = betas.join(estructura[["B/E"]])
comp["beta_U"] = comp["beta_L"] / (1 + (1 - TAU) * comp["B/E"])
comp

In [ ]:
beta_U_mediana = comp["beta_U"].median()
beta_U_promedio = comp["beta_U"].mean()
beta_U_ponderado = np.average(comp["beta_U"], weights=estructura["E"] + estructura["B"])

display(Markdown(f"- **Mediana** de $\\beta^U$: {beta_U_mediana:.3f}\n"
                 f"- **Promedio simple** de $\\beta^U$: {beta_U_promedio:.3f}\n"
                 f"- **Promedio ponderado por valor de la firma**: {beta_U_ponderado:.3f}"))

BETA_U = beta_U_mediana   # usamos la mediana: es robusta a valores extremos

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(comp))
ax.bar(x - 0.2, comp["beta_L"], 0.4, label=r"$\beta^L$ (apalancado)", color="#780404")
ax.bar(x + 0.2, comp["beta_U"], 0.4, label=r"$\beta^U$ (desapalancado)", color="#E37D00")
ax.axhline(BETA_U, color="black", ls="--", lw=1, label=fr"Mediana $\beta^U$ = {BETA_U:.2f}")
ax.set_xticks(x)
ax.set_xticklabels(comp.index)
ax.set_title("Betas de las comparables: apalancado vs. desapalancado")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.show()

**¿Por qué la mediana?** Con pocas comparables, un solo valor extremo puede mover mucho el promedio. Aquí hay dos:

- **Wingstop** tiene un $\beta^U$ muy alto (acción de crecimiento con alta volatilidad).
- **Wendy's** tiene un $\beta^U$ muy bajo. Con $B/E > 3$, su deuda **no es libre de riesgo**, y la fórmula de Hamada (que supone $\beta_B = 0$) le atribuye todo el riesgo al patrimonio, subestimando el beta de activos. Volveremos a esto en la sección de extensiones.

La mediana es menos sensible a estos dos casos.

## Paso 3: Reapalancar con la estructura de capital de Chick-fil-A

Chick-fil-A es conocida por su financiamiento **conservador**. Suponemos una estructura de capital **objetivo** de $B/(B+E) = 20\%$, es decir:
$$
\frac{B}{E} = \frac{0{,}20}{0{,}80} = 0{,}25
$$

Reapalancamos el beta de activos del sector:
$$
\beta^L_{\text{Chick-fil-A}} = \beta^U\left[1 + (1-\tau_c)\frac{B}{E}\right]
$$

In [ ]:
BE_OBJ = W_B_OBJ / (1 - W_B_OBJ)
BETA_L_OBJ = BETA_U * (1 + (1 - TAU) * BE_OBJ)
print(f"B/E objetivo          : {BE_OBJ:.3f}")
print(f"Beta de activos (β^U) : {BETA_U:.3f}")
print(f"Beta reapalancado (β^L): {BETA_L_OBJ:.3f}")

## Paso 4: Tasa libre de riesgo y prima por riesgo de mercado

**Tasa libre de riesgo.** Usamos el rendimiento del **bono del Tesoro de EE.UU. a 10 años** (serie `DGS10` de FRED, Banco de la Reserva Federal de St. Louis). El plazo largo es consistente con el horizonte de los proyectos que se evalúan.

**Prima por riesgo de mercado (MRP).** Usamos el **promedio histórico** del factor `Mkt-RF` de Fama-French (sitio de **Kenneth French**), desde 1926. Es el mismo factor que usamos en el notebook de Fama-French.

> **Advertencia:** `Mkt-RF` es el exceso de retorno del mercado sobre las **letras del Tesoro de corto plazo**, por lo que es algo mayor que la prima sobre el bono a 10 años. Una alternativa muy usada en la práctica es la **prima implícita** que publica Aswath Damodaran cada mes; puede ingresarla en `MRP_MANUAL`.

In [ ]:
try:
    dgs10 = web.DataReader("DGS10", "fred", start="2025-01-01").dropna()
    RF = dgs10.iloc[-1, 0] / 100
    fecha_rf = dgs10.index[-1]
    print(f"Bono del Tesoro a 10 años al {fecha_rf:%d-%m-%Y}: {RF:.2%}")
except Exception as e:
    RF = 0.0511   # valor al 23-sep-2026; actualícelo si la descarga falla
    print(f"No se pudo descargar FRED ({e}). Se usa r_f = {RF:.2%}")

In [ ]:
MRP_MANUAL = None   # por ejemplo 0.045 para usar la prima implícita de Damodaran

ff = web.DataReader("F-F_Research_Data_Factors", "famafrench", start="1926-01-01")[0]
MRP_HIST = ff["Mkt-RF"].mean() * 12 / 100
print(f"Prima histórica Mkt-RF ({ff.index[0]} a {ff.index[-1]}): {MRP_HIST:.2%} anual")

MRP = MRP_MANUAL if MRP_MANUAL is not None else MRP_HIST
print(f"Prima por riesgo de mercado utilizada: {MRP:.2%}")

## Paso 5: Costo del patrimonio

$$
r_E = r_f + \beta^L_{\text{Chick-fil-A}}\,\text{MRP}
$$

In [ ]:
R_E = RF + BETA_L_OBJ * MRP
print(f"r_E = {RF:.2%} + {BETA_L_OBJ:.3f} × {MRP:.2%} = {R_E:.2%}")

## Paso 6: Costo de la deuda con un rating sintético

Chick-fil-A no tiene bonos que transen en bolsa, así que no observamos directamente su tasa de endeudamiento. Usamos el método del **rating sintético** de Aswath Damodaran:

1. Se calcula la **cobertura de intereses**: $\text{Cobertura} = \dfrac{EBIT}{\text{Gastos por intereses}}$.
2. La cobertura se asocia a un **rating** (AAA, AA, …) según una tabla construida con empresas que sí tienen rating.
3. Cada rating tiene un **spread** sobre la tasa libre de riesgo, observado en bonos transados: $r_B = r_f + \text{spread}$.

La tabla siguiente es la de Damodaran para **grandes empresas no financieras, enero 2026**.

Como no conocemos el EBIT de Chick-fil-A, suponemos $EBIT/V = 5\%$ (parámetro `EBIT_V`). Así:
$$
\text{Cobertura} = \frac{EBIT}{r_B\,B} = \frac{EBIT/V}{r_B \cdot B/V}
$$
Como $r_B$ depende del rating y el rating depende de $r_B$, resolvemos con unas pocas **iteraciones**.

In [ ]:
# Damodaran, "Ratings, Interest Coverage Ratios and Default Spread", enero 2026
tabla_rating = pd.DataFrame([
    (-np.inf, 0.20, "D2/D",    0.1900),
    (0.20,   0.65, "C2/C",     0.1600),
    (0.65,   0.80, "Ca2/CC",   0.1261),
    (0.80,   1.25, "Caa/CCC",  0.0885),
    (1.25,   1.50, "B3/B-",    0.0509),
    (1.50,   1.75, "B2/B",     0.0321),
    (1.75,   2.00, "B1/B+",    0.0275),
    (2.00,   2.25, "Ba2/BB",   0.0184),
    (2.25,   2.50, "Ba1/BB+",  0.0138),
    (2.50,   3.00, "Baa2/BBB", 0.0111),
    (3.00,   4.25, "A3/A-",    0.0089),
    (4.25,   5.50, "A2/A",     0.0078),
    (5.50,   6.50, "A1/A+",    0.0070),
    (6.50,   8.50, "Aa2/AA",   0.0055),
    (8.50,  np.inf, "Aaa/AAA", 0.0040),
], columns=["Cobertura desde", "Cobertura hasta", "Rating", "Spread"])

def rating_sintetico(cobertura):
    if np.isinf(cobertura):          # sin deuda: mejor rating posible
        fila = tabla_rating.iloc[-1]
        return fila["Rating"], fila["Spread"]
    fila = tabla_rating[(tabla_rating["Cobertura desde"] <= cobertura) &
                        (cobertura < tabla_rating["Cobertura hasta"])].iloc[0]
    return fila["Rating"], fila["Spread"]

def costo_deuda(w_B, rf=None, ebit_v=None, iteraciones=30):
    rf = RF if rf is None else rf
    ebit_v = EBIT_V if ebit_v is None else ebit_v
    if w_B == 0:
        rating, spread = rating_sintetico(np.inf)
        return rating, rf + spread, np.inf
    r_B = rf
    for _ in range(iteraciones):
        cobertura = ebit_v / (r_B * w_B)
        rating, spread = rating_sintetico(cobertura)
        r_B = rf + spread
    return rating, r_B, cobertura

tabla_rating

In [ ]:
RATING, R_B, COBERTURA = costo_deuda(W_B_OBJ)
print(f"Cobertura de intereses : {COBERTURA:.2f} veces")
print(f"Rating sintético       : {RATING}")
print(f"Costo de la deuda r_B  : {R_B:.2%}  (antes de impuestos)")
print(f"Costo después de impuestos r_B(1-τ): {R_B * (1 - TAU):.2%}")

## Paso 7: El WACC de Chick-fil-A

$$
WACC = \frac{E}{B+E}\,r_E + \frac{B}{B+E}\,r_B\,(1-\tau_c)
$$

In [ ]:
W_E_OBJ = 1 - W_B_OBJ
WACC = W_E_OBJ * R_E + W_B_OBJ * R_B * (1 - TAU)

resumen = pd.DataFrame({"Valor": [
    RF, MRP, BETA_U, BE_OBJ, BETA_L_OBJ, R_E, R_B, TAU, W_B_OBJ, WACC]},
    index=["Tasa libre de riesgo (r_f)", "Prima por riesgo de mercado",
           "Beta de activos (mediana comparables)", "B/E objetivo",
           "Beta reapalancado", "Costo del patrimonio (r_E)",
           f"Costo de la deuda (r_B, rating {RATING})", "Tasa de impuestos",
           "B/(B+E) objetivo", "WACC"])
resumen

Este es el costo de capital con el que Chick-fil-A debería descontar los **flujos de caja libres** de un proyecto de **riesgo similar al de su negocio actual** (por ejemplo, abrir nuevos restaurantes en EE.UU.).

Observe que el WACC es **menor** que $r_E$: la deuda es más barata que el patrimonio y, además, sus intereses generan un **escudo fiscal**.

## Sensibilidad: WACC y estructura de capital

¿Qué pasa si Chick-fil-A se endeuda más? Repetimos el cálculo para distintos niveles de $B/(B+E)$:

- Más deuda ⇒ mayor $\beta^L$ ⇒ mayor $r_E$ (Proposición II de M\&M).
- Más deuda ⇒ más **escudo fiscal** ⇒ el WACC tiende a bajar.
- Pero más deuda ⇒ menor cobertura ⇒ **peor rating** ⇒ mayor $r_B$.

El resultado conecta con el **trade-off** entre escudo fiscal y costos de quiebra que vimos en la presentación (enfoque de Leland).

In [ ]:
filas = []
for w_B in np.arange(0, 0.61, 0.025):
    be = w_B / (1 - w_B)
    b_L = BETA_U * (1 + (1 - TAU) * be)
    r_E = RF + b_L * MRP
    rating, r_B, cob = costo_deuda(w_B)
    wacc = (1 - w_B) * r_E + w_B * r_B * (1 - TAU)
    filas.append({"B/(B+E)": w_B, "beta_L": b_L, "r_E": r_E,
                  "Rating": rating, "r_B": r_B, "WACC": wacc})
sens = pd.DataFrame(filas)

optimo = sens.loc[sens["WACC"].idxmin()]
print(f"WACC mínimo: {optimo['WACC']:.2%} con B/(B+E) = {optimo['B/(B+E)']:.1%} (rating {optimo['Rating']})")
sens.style.format({"B/(B+E)": "{:.1%}", "beta_L": "{:.3f}", "r_E": "{:.2%}",
                   "r_B": "{:.2%}", "WACC": "{:.2%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sens["B/(B+E)"], sens["r_E"], color="#780404", lw=2, label=r"$r_E$")
ax.plot(sens["B/(B+E)"], sens["r_B"] * (1 - TAU), color="#1f4e9c", lw=2, label=r"$r_B(1-\tau_c)$")
ax.plot(sens["B/(B+E)"], sens["WACC"], color="#E37D00", lw=3, label="WACC")
ax.axvline(W_B_OBJ, color="gray", ls=":", label="Estructura objetivo")
ax.scatter(optimo["B/(B+E)"], optimo["WACC"], color="black", zorder=5, label="WACC mínimo")
ax.set_xlabel("B / (B + E)")
ax.set_ylabel("Tasa anual")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title(f"{EMPRESA}: costo de capital según endeudamiento")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

**Lectura del gráfico.**

- Con poca deuda, el WACC **baja** a medida que aumenta $B/(B+E)$: domina el escudo fiscal.
- Cuando la cobertura de intereses cae, el **rating empeora** y el costo de la deuda salta. A partir de ese punto el WACC **sube**.
- Existe entonces un rango de endeudamiento que **minimiza** el WACC, coherente con la teoría del *trade-off*.

**Limitación.** En este ejercicio $r_E$ se calcula con Hamada, que supone deuda libre de riesgo. Con niveles altos de deuda, parte del riesgo lo asumen los acreedores y la fórmula deja de ser precisa.

## Extensión: ¿y si la deuda de las comparables es riesgosa?

Cuando la deuda es riesgosa ($\beta_B > 0$), la relación entre betas es (ver presentación):
$$
\beta^L = \beta^U + (\beta^U - \beta_B)(1-\tau_c)\frac{B}{E}
\qquad\Longrightarrow\qquad
\beta^U = \frac{\beta^L + \beta_B(1-\tau_c)\,B/E}{1 + (1-\tau_c)\,B/E}
$$

Un valor razonable para $\beta_B$ se obtiene del CAPM aplicado a la deuda: si el spread es de 1–2% y la prima de mercado es cercana a 8%, entonces $\beta_B \approx \text{spread}/\text{MRP} \approx 0{,}1\text{–}0{,}25$.

Veamos cómo cambian los betas de activos con $\beta_B = 0{,}25$ para todas las comparables.

In [ ]:
BETA_B = 0.25
comp["beta_U (B riesgosa)"] = ((comp["beta_L"] + BETA_B * (1 - TAU) * comp["B/E"])
                                / (1 + (1 - TAU) * comp["B/E"]))
display(comp[["beta_L", "B/E", "beta_U", "beta_U (B riesgosa)"]])
print(f"Mediana β^U con deuda libre de riesgo: {comp['beta_U'].median():.3f}")
print(f"Mediana β^U con deuda riesgosa (β_B = {BETA_B}): {comp['beta_U (B riesgosa)'].median():.3f}")

El ajuste es más importante para las empresas **más endeudadas**, especialmente Wendy's. Con deuda riesgosa, su beta de activos se vuelve mucho más parecido al del resto del sector.

## Exportar tablas a LaTeX

Las siguientes celdas generan las tablas para la presentación en Overleaf (`Comparables_WACC.tex` y `WACC_Resumen.tex`).

In [ ]:
def tabla_latex(filas, encabezado, alineacion, titulo):
    lineas = [r"\begin{table}[!htbp] \centering \scriptsize",
              rf"\caption{{{titulo}}}",
              rf"\begin{{tabular}}{{{alineacion}}}",
              r"\toprule",
              " & ".join(encabezado) + r" \\",
              r"\midrule"]
    lineas += [" & ".join(f) + r" \\" for f in filas]
    lineas += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lineas)

num = lambda x, d=3: f"{x:.{d}f}".replace(".", "{,}")
pct = lambda x: f"{100 * x:.2f}".replace(".", "{,}") + r"\%"

filas_comp = [[t, NOMBRES[t], num(comp.loc[t, "beta_L"]), f"({num(comp.loc[t, 'e.e.'])})",
               num(comp.loc[t, "R2"]), num(comp.loc[t, "B/E"]), num(comp.loc[t, "beta_U"])]
              for t in TICKERS]
tex_comp = tabla_latex(
    filas_comp,["Ticker", "Empresa", r"$\hat\beta^L$", "e.e.", "$R^2$", "$B/E$", r"$\beta^U$"],
    "llccccc", f"Betas de las comparables ({retornos.index[0]:%Y-%m} a {retornos.index[-1]:%Y-%m})")
tex_comp = tex_comp.replace(r"\bottomrule",
    r"\midrule" + "\n" + rf"\multicolumn{{6}}{{l}}{{\textbf{{Mediana}}}} & \textbf{{{num(BETA_U)}}} \\" + "\n" + r"\bottomrule")

filas_res = [["Tasa libre de riesgo, $r_f$", pct(RF)],
             ["Prima por riesgo de mercado", pct(MRP)],
             [r"Beta de activos, $\beta^U$", num(BETA_U)],
             ["$B/(B+E)$ objetivo", pct(W_B_OBJ)],
             [r"Beta reapalancado, $\beta^L$", num(BETA_L_OBJ)],
             ["Costo del patrimonio, $r_E$", pct(R_E)],
             [f"Costo de la deuda, $r_B$ (rating {RATING})", pct(R_B)],
             [r"Tasa de impuestos, $\tau_c$", pct(TAU)],
             [r"\textbf{WACC}", rf"\textbf{{{pct(WACC)}}}"]]
tex_res = tabla_latex(filas_res, ["Componente", "Valor"], "lr", f"WACC de {EMPRESA}")

for nombre, contenido in [("Comparables_WACC.tex", tex_comp), ("WACC_Resumen.tex", tex_res)]:
    with open(nombre, "w", encoding="utf-8") as f:
        f.write(contenido)
    print(contenido, "\n")
print("Tablas exportadas: Comparables_WACC.tex, WACC_Resumen.tex")

## Discusión: ¿qué tan confiable es este WACC?

1. **Elección de comparables.** Es la decisión más importante. Chick-fil-A opera solo en EE.UU., casi sin locales propios y con crecimiento sostenido; ninguna comparable es idéntica.
2. **Error de estimación.** Los errores estándar de los betas son grandes (0,13 a 0,44). Promediar varias comparables reduce ese error.
3. **Estructura de capital objetivo.** El 20% es un supuesto. Lo relevante es la estructura **objetivo de largo plazo**, no la de un año en particular.
4. **Prima por riesgo de mercado.** Pasar de la prima histórica a la implícita puede mover el WACC en más de un punto porcentual. Pruebe con `MRP_MANUAL`.
5. **Empresa privada.** El CAPM supone que el dueño está **diversificado**. Los dueños de una empresa familiar suelen no estarlo, y además sus acciones son **ilíquidas**. En la práctica, muchos analistas agregan una **prima por iliquidez** o usan un *beta total*.

### Preguntas para el alumno
- ¿Cómo cambia el WACC si excluye a Wingstop y Wendy's de las comparables?
- ¿Qué pasa si usa la prima implícita de Damodaran en vez de la histórica?
- Si Chick-fil-A quisiera financiar su expansión con 40% de deuda, ¿qué rating obtendría y cuál sería su WACC?